# Neuro-Symbolic Disease Prediction Model Training

## Overview

This notebook implements a memory-efficient neuro-symbolic disease prediction system that combines neural networks with medical knowledge graph reasoning for accurate disease diagnosis.

## Key Features

### Model Architecture
1. **Medical Knowledge Graph**: Neo4j-based ontology with TF-IDF and PMI weighted disease-symptom relationships extracted from medical datasets. Provides evidence-based symbolic reasoning with sensitivity and specificity metrics.

2. **Neural Network Components**:
   - EfficientNeuralPredictor: Lightweight neural network with LayerNorm and efficient initialization
   - LightweightFusion: Reduced-parameter fusion network for neural-symbolic integration
   - SymbolicPredictionCache: LRU cache for knowledge graph predictions

3. **Semantic Symptom Matching**: Uses sentence transformers to match colloquial symptom terms to medical vocabulary, enabling real-world usability with common language inputs.

### Performance Improvements
1. **Mixed Precision Training (FP16)**: Reduces memory usage by approximately 50% and increases training speed by 2-3x on GPU
2. **Gradient Accumulation**: Enables effective larger batch sizes without out-of-memory errors
3. **Symbolic Prediction Caching**: Pre-computes and caches knowledge graph queries to avoid redundant computations during training
4. **Streamlined Architecture**: Reduced from [256, 128, 64] to [128, 64] hidden layers for efficiency
5. **LayerNorm instead of BatchNorm**: More memory-efficient normalization
6. **Efficient Data Loading**: Batch processing with progress monitoring

### Expected Performance
- **Training Speed**: 2-3x faster with mixed precision
- **Memory Usage**: Approximately 50% reduction with FP16 and reduced architecture
- **Batch Size**: Increased from 32 to 64 with gradient accumulation
- **Effective Batch Size**: 256 (64 batch size × 4 accumulation steps)

## Training Configuration

- **Data Source**: Neo4j medical ontology via ngrok tunnel
- **Training Strategy**: End-to-end training with balanced loss function
- **Loss Function**: Multi-component loss combining neural and symbolic predictions (0.5/0.5 balance)
- **Optimization**: AdamW optimizer with cosine annealing learning rate scheduler
- **Mixed Precision**: FP16 automatic mixed precision with gradient scaling
- **Gradient Accumulation**: 4 steps for effective batch size of 256
- **Model Persistence**: Saved to local disk and Google Drive

## Expected Outcomes

The trained model provides:
- Disease predictions with probability scores (softmax normalized)
- Fusion weights showing neural vs symbolic contribution balance
- Comprehensive medical validation metrics (accuracy, precision, recall, F1, sensitivity, specificity)
- Semantic symptom matching for real-world input handling
- Evidence-based explanations for predictions
- Comparison between fused model and neural-only baseline

## Real-World Usability

The model includes semantic symptom matching that allows users to input symptoms in colloquial language (e.g., "cough", "fever", "headache") which are automatically matched to medical terminology in the training vocabulary using sentence transformer embeddings.

---
## Section 1: Environment Setup

Mount Google Drive for model persistence and import required dependencies.

In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install pandas numpy scikit-learn
!pip install networkx
!pip install neo4j
!pip install sentence-transformers

Looking in indexes: https://download.pytorch.org/whl/cu118
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.8/325.8 kB 10.3 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer, StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
import json
from typing import Dict, List, Tuple, Set, Optional
import networkx as nx
from collections import defaultdict, Counter
import logging
import warnings
from dataclasses import dataclass
import pickle
from pathlib import Path
from neo4j import GraphDatabase
import datetime
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Neo4j Configuration
NEO4J_URI = "bolt://8.tcp.ngrok.io:16633"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "password"

print(f"Training started at: {datetime.datetime.now()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Training started at: 2025-10-12 18:09:35.959213
PyTorch version: 2.8.0+cu126
CUDA available: True
GPU: Tesla T4
GPU Memory: 15.83 GB


---
## Section 2: Data Structures

Define core data classes for medical rules and validation metrics.

In [4]:
@dataclass
class MedicalRule:
    """Represents a validated medical rule with evidence."""
    symptoms: frozenset
    disease: str
    confidence: float
    evidence_source: str
    sensitivity: float = 0.0
    specificity: float = 0.0
    prevalence: float = 0.0

@dataclass
class ValidationMetrics:
    """Comprehensive validation metrics for medical AI evaluation."""
    accuracy: float
    precision: float
    recall: float
    f1_score: float
    auc_roc: float
    sensitivity: float
    specificity: float
    ppv: float  # Positive Predictive Value
    npv: float  # Negative Predictive Value

---
## Section 3: Cache and Medical Knowledge Graph

Define cache structure and symbolic reasoning component that loads disease-symptom relationships from Neo4j ontology.

In [5]:
class SymbolicPredictionCache:
    """Cache for symbolic predictions to avoid recomputation."""

    def __init__(self, max_size: int = 10000):
        self.cache = {}
        self.max_size = max_size

    def get(self, key):
        return self.cache.get(key)

    def set(self, key, value):
        if len(self.cache) >= self.max_size:
            self.cache.pop(next(iter(self.cache)))
        self.cache[key] = value

    def clear(self):
        self.cache.clear()

logger.info("SymbolicPredictionCache class loaded")

In [6]:
class MedicalKnowledgeGraph:
    """Medical knowledge graph for evidence-based disease reasoning with caching."""

    def __init__(self, neo4j_uri=NEO4J_URI, neo4j_user=NEO4J_USER, neo4j_password=NEO4J_PASSWORD):
        self.graph = nx.MultiDiGraph()
        self.symptom_disease_rules = {}
        self.differential_diagnoses = defaultdict(set)
        self.contraindications = defaultdict(set)
        self.neo4j_uri = neo4j_uri
        self.neo4j_user = neo4j_user
        self.neo4j_password = neo4j_password
        self.driver = None
        self.prediction_cache = SymbolicPredictionCache(max_size=10000)
        self.connect_to_neo4j()

    def connect_to_neo4j(self):
        """Establish connection to Neo4j database."""
        try:
            self.driver = GraphDatabase.driver(self.neo4j_uri, auth=(self.neo4j_user, self.neo4j_password))
            with self.driver.session() as session:
                session.run("RETURN 1")
            logger.info("Neo4j connection established")
        except Exception as e:
            logger.error("Neo4j connection failed: %s", str(e))
            self.driver = None

    def add_validated_rule(self, rule: MedicalRule):
        """Add a medically validated rule to the knowledge base."""
        rule_key = rule.symptoms
        if rule_key not in self.symptom_disease_rules:
            self.symptom_disease_rules[rule_key] = []
        self.symptom_disease_rules[rule_key].append(rule)
        for symptom in rule.symptoms:
            self.graph.add_edge(symptom, rule.disease, weight=rule.confidence,
                              evidence=rule.evidence_source, sensitivity=rule.sensitivity,
                              specificity=rule.specificity)

    def get_evidence_based_predictions(self, symptoms: Set[str], patient_demographics: Dict = None) -> Tuple[Dict[str, float], Dict[str, List]]:
        """Generate disease predictions using evidence-based rules with caching."""
        cache_key = frozenset(symptoms)
        cached = self.prediction_cache.get(cache_key)
        if cached is not None:
            return cached

        disease_scores = defaultdict(float)
        disease_evidence = defaultdict(list)

        for rule_symptoms, rules in self.symptom_disease_rules.items():
            if rule_symptoms.issubset(symptoms):
                for rule in rules:
                    evidence_weight = rule.sensitivity * rule.specificity * (rule.prevalence + 0.1)
                    weighted_score = rule.confidence * evidence_weight
                    disease_scores[rule.disease] += weighted_score
                    disease_evidence[rule.disease].append({
                        'rule': rule, 'weight': weighted_score, 'evidence': rule.evidence_source
                    })

        for disease in list(disease_scores.keys()):
            if disease in self.contraindications:
                for contraindicated_symptom, reason in self.contraindications[disease]:
                    if contraindicated_symptom in symptoms:
                        disease_scores[disease] *= 0.1

        total_score = sum(disease_scores.values())
        if total_score > 0:
            disease_scores = {k: v/total_score for k, v in disease_scores.items()}

        result = (dict(disease_scores), dict(disease_evidence))
        self.prediction_cache.set(cache_key, result)
        return result

    def load_from_neo4j(self, min_weight=0.3, max_rules_per_disease=50):
        """Load medical rules from Neo4j ontology database."""
        if not self.driver:
            logger.warning("No Neo4j connection available")
            return False

        try:
            logger.info("Loading disease-symptom relationships (min_weight=%.2f)", min_weight)
            with self.driver.session() as session:
                result = session.run("""
                    MATCH (d:Disease)-[r:HAS_SYMPTOM]->(s:Symptom)
                    WHERE r.weight >= $min_weight
                    RETURN d.name as disease_name, s.name as symptom_name,
                           r.weight as weight, r.tfidf_weight as tfidf_weight,
                           r.pmi_confidence as pmi_confidence
                    ORDER BY d.name, r.weight DESC
                """, min_weight=min_weight)

                disease_symptoms = defaultdict(list)
                total_relationships = 0
                for record in result:
                    disease_key = record['disease_name']
                    symptom_info = {
                        'name': record['symptom_name'],
                        'weight': record['weight'],
                        'tfidf_weight': record['tfidf_weight'],
                        'pmi_confidence': record['pmi_confidence']
                    }
                    disease_symptoms[disease_key].append(symptom_info)
                    total_relationships += 1

                logger.info("Loaded %d disease-symptom relationships", total_relationships)
                logger.info("Found %d unique diseases", len(disease_symptoms))

                logger.info("Creating medical rules...")
                rules_created = 0
                for disease_name, symptoms_list in tqdm(disease_symptoms.items(),
                                                        desc="Processing diseases",
                                                        leave=False):
                    symptoms_list.sort(key=lambda x: x['weight'], reverse=True)
                    top_symptoms = symptoms_list[:max_rules_per_disease]
                    self._create_rules_from_symptoms(disease_name, top_symptoms)
                    rules_created += 1

                total_rules = sum(len(rules) for rules in self.symptom_disease_rules.values())
                logger.info("Created %d medical rules from %d diseases", total_rules, rules_created)
                return True
        except Exception as e:
            logger.error("Failed to load from Neo4j: %s", str(e))
            return False

    def _create_rules_from_symptoms(self, disease_name, symptoms_list):
        """Create medical rules from symptom data with proper weighting."""
        for symptom_info in symptoms_list:
            if symptom_info['weight'] >= 0.4:
                rule = MedicalRule(
                    symptoms=frozenset([symptom_info['name']]),
                    disease=disease_name,
                    confidence=float(symptom_info['weight']),
                    evidence_source="Neo4j_Ontology_TF_IDF_PMI",
                    sensitivity=min(0.95, float(symptom_info['tfidf_weight']) * 1.2),
                    specificity=min(0.95, float(symptom_info['pmi_confidence']) * 1.1),
                    prevalence=0.1
                )
                self.add_validated_rule(rule)

        high_weight_symptoms = [s for s in symptoms_list if s['weight'] >= 0.5]
        for i in range(len(high_weight_symptoms)):
            for j in range(i + 1, min(len(high_weight_symptoms), i + 4)):
                symptom1, symptom2 = high_weight_symptoms[i], high_weight_symptoms[j]
                combined_weight = (symptom1['weight'] + symptom2['weight']) / 2
                rule = MedicalRule(
                    symptoms=frozenset([symptom1['name'], symptom2['name']]),
                    disease=disease_name,
                    confidence=min(0.95, combined_weight * 1.1),
                    evidence_source="Neo4j_Ontology_Combined",
                    sensitivity=min(0.92, combined_weight * 1.1),
                    specificity=min(0.93, (symptom1['pmi_confidence'] + symptom2['pmi_confidence']) / 2 * 1.05),
                    prevalence=0.08
                )
                self.add_validated_rule(rule)

    def close(self):
        """Close Neo4j connection."""
        if self.driver:
            self.driver.close()
            logger.info("Neo4j connection closed")

logger.info("MedicalKnowledgeGraph class loaded with caching support")

---
## Section 4: Neural Network Components

Define the neural network architectures for disease prediction and evidence-based fusion.

In [7]:
class SymptomAttention(nn.Module):
    """Multi-head attention mechanism for symptom importance weighting."""

    def __init__(self, input_dim: int, num_heads: int = 8, dropout: float = 0.1):
        super(SymptomAttention, self).__init__()
        self.num_heads = num_heads
        self.head_dim = input_dim // num_heads
        assert input_dim % num_heads == 0, "input_dim must be divisible by num_heads"

        self.query = nn.Linear(input_dim, input_dim)
        self.key = nn.Linear(input_dim, input_dim)
        self.value = nn.Linear(input_dim, input_dim)
        self.output = nn.Linear(input_dim, input_dim)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(input_dim)

    def forward(self, x):
        batch_size = x.size(0)
        residual = x

        # Multi-head attention
        Q = self.query(x).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.key(x).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.value(x).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attention = F.softmax(scores, dim=-1)
        attention = self.dropout(attention)

        # Apply attention to values
        context = torch.matmul(attention, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.head_dim)

        # Output projection with residual connection
        output = self.output(context)
        output = self.dropout(output)
        output = self.layer_norm(output + residual)

        return output.squeeze(1)


class ResidualBlock(nn.Module):
    """Residual block with layer normalization and gating."""

    def __init__(self, dim: int, dropout: float = 0.2):
        super(ResidualBlock, self).__init__()
        self.linear1 = nn.Linear(dim, dim)
        self.linear2 = nn.Linear(dim, dim)
        self.gate = nn.Linear(dim, dim)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x

        # First transformation
        out = self.linear1(x)
        out = self.norm1(out)
        out = F.gelu(out)
        out = self.dropout(out)

        # Second transformation
        out = self.linear2(out)
        out = self.norm2(out)

        # Gating mechanism
        gate = torch.sigmoid(self.gate(x))
        out = gate * out + (1 - gate) * residual

        return out


class EnhancedNeuralPredictor(nn.Module):
    """Enhanced neural network with attention, residual connections, and uncertainty."""

    def __init__(self, input_dim: int, hidden_dims: List[int], output_dim: int,
                 num_attention_heads: int = 8, dropout_rate: float = 0.3):
        super(EnhancedNeuralPredictor, self).__init__()

        # Input projection with attention
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]),
            nn.LayerNorm(hidden_dims[0]),
            nn.GELU(),
            nn.Dropout(dropout_rate)
        )

        # Symptom attention mechanism
        self.symptom_attention = SymptomAttention(
            hidden_dims[0],
            num_heads=num_attention_heads,
            dropout=dropout_rate
        )

        # Residual blocks for deep feature extraction
        self.residual_blocks = nn.ModuleList()
        for i in range(len(hidden_dims) - 1):
            if hidden_dims[i] == hidden_dims[i+1]:
                self.residual_blocks.append(
                    ResidualBlock(hidden_dims[i], dropout=dropout_rate)
                )
            else:
                # Transition layer when dimensions change
                self.residual_blocks.append(
                    nn.Sequential(
                        nn.Linear(hidden_dims[i], hidden_dims[i+1]),
                        nn.LayerNorm(hidden_dims[i+1]),
                        nn.GELU(),
                        nn.Dropout(dropout_rate)
                    )
                )

        # Output layers
        final_dim = hidden_dims[-1]
        self.output_layer = nn.Linear(final_dim, output_dim)

        self._init_weights()

    def _init_weights(self):
        """Xavier/Glorot initialization for better convergence."""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x, return_uncertainty=False):
        """Forward pass with optional uncertainty estimation."""
        # Input projection
        x = self.input_proj(x)

        # Add sequence dimension for attention
        x = x.unsqueeze(1)

        # Apply symptom attention
        x = self.symptom_attention(x)

        # Pass through residual blocks
        for block in self.residual_blocks:
            x = block(x)

        # Output logits
        logits = self.output_layer(x)

        if return_uncertainty:
            # Monte Carlo dropout for uncertainty
            self.train()
            predictions = [torch.softmax(logits, dim=1) for _ in range(10)]
            self.eval()
            predictions = torch.stack(predictions)
            return predictions.mean(dim=0), predictions.var(dim=0)

        return logits


class CrossAttentionFusion(nn.Module):
    """Advanced fusion with cross-attention between neural and symbolic predictions."""

    def __init__(self, input_dim: int, num_diseases: int, hidden_dim: int = 256):
        super(CrossAttentionFusion, self).__init__()

        # Context encoder
        self.context_encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU()
        )

        # Prediction encoders
        self.neural_encoder = nn.Sequential(
            nn.Linear(num_diseases, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU()
        )

        self.symbolic_encoder = nn.Sequential(
            nn.Linear(num_diseases, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU()
        )

        # Cross-attention between neural and symbolic
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=hidden_dim // 2,
            num_heads=4,
            dropout=0.1,
            batch_first=True
        )

        # Fusion weight predictor
        fusion_input_dim = hidden_dim // 2 * 3
        self.fusion_weights_net = nn.Sequential(
            nn.Linear(fusion_input_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Linear(64, 3)  # neural_weight, symbolic_weight, confidence
        )

        # Final output projection
        self.output_proj = nn.Sequential(
            nn.Linear(num_diseases, num_diseases),
            nn.LayerNorm(num_diseases)
        )

    def forward(self, symptoms, neural_pred, symbolic_pred):
        """Dynamically fuse predictions using cross-attention."""
        # Encode context and predictions
        context = self.context_encoder(symptoms)
        neural_enc = self.neural_encoder(neural_pred)
        symbolic_enc = self.symbolic_encoder(symbolic_pred)

        # Cross-attention between neural and symbolic
        neural_att, _ = self.cross_attention(
            neural_enc.unsqueeze(1),
            symbolic_enc.unsqueeze(1),
            symbolic_enc.unsqueeze(1)
        )
        neural_att = neural_att.squeeze(1)

        # Concatenate for fusion weight prediction
        fusion_input = torch.cat([context, neural_att, symbolic_enc], dim=1)
        fusion_logits = self.fusion_weights_net(fusion_input)

        # Compute fusion weights with minimum contribution
        raw_weights = F.softmax(fusion_logits[:, :2], dim=1)
        min_weight = 0.2
        weights = torch.clamp(raw_weights, min=min_weight)
        weights = weights / weights.sum(dim=1, keepdim=True)

        # Extract confidence
        confidence = torch.sigmoid(fusion_logits[:, 2:3])

        # Fuse predictions
        fused = weights[:, 0:1] * neural_pred + weights[:, 1:2] * symbolic_pred
        fused = self.output_proj(fused)

        # Create fusion weights for reporting
        fusion_weights = torch.cat([weights, confidence], dim=1)

        return fused, confidence, fusion_weights


logger.info("Enhanced neural network classes loaded")

In [8]:
class SemanticSymptomMatcher:
    """
    Uses semantic similarity to match user symptoms to medical vocabulary.

    This allows users to input colloquial terms like "cough" or "fever"
    and automatically find the closest matches in the medical terminology.
    """

    def __init__(self, symptoms_vocab: List[str]):
        logger.info("Loading semantic model for symptom matching...")
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.symptoms_vocab = list(symptoms_vocab)

        # Pre-compute embeddings for all vocabulary symptoms
        logger.info("Computing embeddings for %d symptoms...", len(self.symptoms_vocab))
        self.vocab_embeddings = self.model.encode(
            self.symptoms_vocab,
            show_progress_bar=True,
            convert_to_numpy=True
        )
        logger.info("Semantic matcher initialized")

    def find_matches(self, user_symptom: str, threshold: float = 0.5, top_k: int = 3) -> List[Tuple[str, float]]:
        """
        Find semantically similar symptoms from vocabulary.

        Args:
            user_symptom: User's input symptom (e.g., "cough")
            threshold: Minimum similarity score (0-1)
            top_k: Maximum number of matches to return

        Returns:
            List of (symptom, similarity_score) tuples
        """
        # Encode user symptom
        user_embedding = self.model.encode([user_symptom], convert_to_numpy=True)

        # Compute cosine similarity
        similarities = cosine_similarity(user_embedding, self.vocab_embeddings)[0]

        # Get top matches above threshold
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        matches = [
            (self.symptoms_vocab[idx], float(similarities[idx]))
            for idx in top_indices
            if similarities[idx] >= threshold
        ]

        return matches

    def match_symptoms_batch(self, user_symptoms: List[str],
                            threshold: float = 0.5,
                            top_k_per_symptom: int = 2) -> Dict[str, List[Tuple[str, float]]]:
        """
        Match multiple symptoms at once.

        Returns:
            Dictionary mapping user symptom -> list of (matched_symptom, score)
        """
        results = {}
        for symptom in user_symptoms:
            symptom_clean = symptom.lower().strip()

            # Check for exact match first
            if symptom_clean in self.symptoms_vocab:
                results[symptom] = [(symptom_clean, 1.0)]
            else:
                # Find semantic matches
                matches = self.find_matches(symptom_clean, threshold, top_k_per_symptom)
                results[symptom] = matches

        return results

logger.info("SemanticSymptomMatcher class loaded")

---
## Section 5: Disease Predictor

Main predictor class that coordinates neural and symbolic components with comprehensive validation.

In [9]:
class DiseasePredictor:
    """Memory-efficient neuro-symbolic disease predictor with mixed precision training."""

    def __init__(self, use_amp=True):
        self.knowledge_graph = MedicalKnowledgeGraph()
        self.neural_model = None
        self.fusion_model = None
        self.symptom_encoder = None
        self.disease_encoder = None
        self.symptom_scaler = StandardScaler()
        self.symptoms_vocab = []
        self.diseases_vocab = []
        self.validation_metrics = {}
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.use_amp = use_amp and torch.cuda.is_available()
        self.scaler = torch.cuda.amp.GradScaler() if self.use_amp else None
        self.symbolic_predictions_cache = None

        print(f"✓ Device: {self.device}, Mixed Precision: {self.use_amp}")

    def load_knowledge_base_from_neo4j(self, min_weight: float = 0.3):
        """Load medical knowledge base from Neo4j ontology."""
        success = self.knowledge_graph.load_from_neo4j(min_weight=min_weight)
        if success:
            print(f"✓ Knowledge base loaded successfully")
        return success

    def _compute_class_weights(self, y):
        """Compute class weights for handling imbalanced datasets."""
        y_np = y.cpu().numpy() if torch.is_tensor(y) else y
        class_counts = np.bincount(y_np)
        weights = len(y_np) / (len(class_counts) * class_counts)
        return torch.FloatTensor(weights).to(self.device)

    def _precompute_symbolic_predictions(self, X_tensor):
        """Pre-compute all symbolic predictions once before training."""
        print("\n" + "─"*60)
        print("Pre-computing symbolic predictions...")
        print("─"*60)

        batch_size = 256
        num_samples = X_tensor.shape[0]
        symbolic_preds = torch.zeros(num_samples, len(self.diseases_vocab), device=self.device)

        with tqdm(total=num_samples, desc="Computing symbolic predictions") as pbar:
            for i in range(0, num_samples, batch_size):
                batch_X = X_tensor[i:i+batch_size]
                symbolic_preds[i:i+batch_size] = self._get_symbolic_predictions_batch_vectorized(batch_X)
                pbar.update(batch_X.shape[0])

        print(f"✓ Pre-computed {num_samples} symbolic predictions")
        print(f"✓ Cache size: {symbolic_preds.element_size() * symbolic_preds.nelement() / 1e6:.2f} MB")
        return symbolic_preds

    def _get_symbolic_predictions_batch_vectorized(self, X_batch):
        """Vectorized batch symbolic prediction."""
        batch_size = X_batch.shape[0]
        symbolic_preds = torch.zeros(batch_size, len(self.diseases_vocab), device=self.device)

        active_mask = X_batch > 0.5
        X_batch_cpu = X_batch.cpu().numpy()
        active_mask_cpu = active_mask.cpu().numpy()

        for i in range(batch_size):
            active_indices = np.where(active_mask_cpu[i])[0]
            if len(active_indices) == 0:
                continue

            active_symptoms = set(self.symptoms_vocab[j] for j in active_indices)
            disease_scores, _ = self.knowledge_graph.get_evidence_based_predictions(active_symptoms)

            for disease, score in disease_scores.items():
                if disease in self.diseases_vocab:
                    symbolic_preds[i, self.diseases_vocab.index(disease)] = score

        return symbolic_preds

    def _get_symbolic_predictions_batch(self, X_batch):
        """Legacy method for compatibility."""
        return self._get_symbolic_predictions_batch_vectorized(X_batch)

    def _compute_metrics(self, y_true, y_pred, y_proba) -> ValidationMetrics:
        """Compute comprehensive metrics."""
        accuracy = accuracy_score(y_true, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)

        try:
            auc_roc = roc_auc_score(y_true, y_proba, multi_class='ovr', average='weighted')
        except ValueError:
            auc_roc = 0.0

        specificities = []
        for class_idx in range(len(self.diseases_vocab)):
            tn = np.sum((y_true != class_idx) & (y_pred != class_idx))
            fp = np.sum((y_true != class_idx) & (y_pred == class_idx))
            specificities.append(tn / (tn + fp) if (tn + fp) > 0 else 0)

        return ValidationMetrics(
            accuracy=accuracy, precision=precision, recall=recall, f1_score=f1,
            auc_roc=auc_roc, sensitivity=recall, specificity=np.mean(specificities),
            ppv=precision, npv=0.0
        )

    def train_model(self, symptoms_data: List[List[str]], diseases_data: List[str],
                    epochs: int = 100, batch_size: int = 64,
                    accumulation_steps: int = 4, learning_rate: float = 0.001):
        """Train neuro-symbolic disease prediction model."""

        print("\n" + "─"*60)
        print("PHASE 1: Data Encoding")
        print("─"*60)
        self.symptom_encoder = MultiLabelBinarizer()
        self.disease_encoder = LabelEncoder()

        X = self.symptom_scaler.fit_transform(self.symptom_encoder.fit_transform(symptoms_data))
        y = self.disease_encoder.fit_transform(diseases_data)
        print(f"✓ Encoded {X.shape[0]} samples with {X.shape[1]} features")

        X_tensor = torch.FloatTensor(X).to(self.device)
        y_tensor = torch.LongTensor(y).to(self.device)
        print(f"✓ Transferred data to {self.device}")

        self.symbolic_predictions_cache = self._precompute_symbolic_predictions(X_tensor)

        print("\n" + "─"*60)
        print("PHASE 2: Model Initialization")
        print("─"*60)
        self.neural_model = EnhancedNeuralPredictor(
            input_dim=X.shape[1],
            hidden_dims=[256, 192, 128],
            num_attention_heads=8,
            output_dim=len(self.diseases_vocab)
        ).to(self.device)

        num_params_neural = sum(p.numel() for p in self.neural_model.parameters())
        print(f"✓ Neural model initialized: {num_params_neural:,} parameters")

        self.fusion_model = CrossAttentionFusion(
            input_dim=X.shape[1],
            num_diseases=len(self.diseases_vocab),
            hidden_dim=256
        ).to(self.device)

        num_params_fusion = sum(p.numel() for p in self.fusion_model.parameters())
        print(f"✓ Fusion model initialized: {num_params_fusion:,} parameters")
        print(f"✓ Total parameters: {num_params_neural + num_params_fusion:,}")

        optimizer = optim.AdamW(
            list(self.neural_model.parameters()) + list(self.fusion_model.parameters()),
            lr=learning_rate,
            weight_decay=1e-5
        )
        print(f"✓ AdamW optimizer configured (lr={learning_rate})")

        criterion = nn.CrossEntropyLoss(weight=self._compute_class_weights(y_tensor))
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        print(f"✓ Loss function and scheduler initialized")

        print("\n" + "─"*60)
        print("PHASE 3: Training")
        print("─"*60)
        print(f"Configuration:")
        print(f"  • Epochs: {epochs}")
        print(f"  • Batch size: {batch_size}")
        print(f"  • Accumulation steps: {accumulation_steps}")
        print(f"  • Effective batch size: {batch_size * accumulation_steps}")
        print(f"  • Mixed precision: {self.use_amp}")
        print(f"  • Symbolic predictions: cached")
        print("─"*60)

        self.neural_model.train()
        self.fusion_model.train()

        num_batches = (len(X_tensor) + batch_size - 1) // batch_size
        best_loss = float('inf')

        for epoch in range(epochs):
            total_loss = 0
            optimizer.zero_grad()

            show_progress = (epoch % 10 == 0)
            batch_iterator = tqdm(range(0, len(X_tensor), batch_size),
                                 desc=f'Epoch {epoch+1}/{epochs}',
                                 disable=not show_progress,
                                 leave=False)

            for batch_idx, i in enumerate(batch_iterator):
                batch_X = X_tensor[i:i+batch_size]
                batch_y = y_tensor[i:i+batch_size]

                symbolic_pred = self.symbolic_predictions_cache[i:i+batch_size]

                if self.use_amp:
                    with torch.cuda.amp.autocast():
                        neural_pred = self.neural_model(batch_X)
                        fused_pred, confidence, fusion_weights = self.fusion_model(batch_X, neural_pred, symbolic_pred)

                        # Balanced loss function (0.5/0.5) with entropy regularization
                        neural_loss = criterion(neural_pred, batch_y)
                        fused_loss = criterion(fused_pred, batch_y)

                        # Add entropy regularization to prevent weight collapse
                        weight_entropy = -torch.mean(
                            fusion_weights[:, 0] * torch.log(fusion_weights[:, 0] + 1e-8) +
                            fusion_weights[:, 1] * torch.log(fusion_weights[:, 1] + 1e-8)
                        )
                        entropy_penalty = 0.1 * (0.693 - weight_entropy)  # 0.693 = max entropy for 2 weights

                        loss = (0.5 * neural_loss + 0.5 * fused_loss + entropy_penalty)
                        loss = loss / accumulation_steps

                    self.scaler.scale(loss).backward()

                    if (batch_idx + 1) % accumulation_steps == 0:
                        self.scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(
                            list(self.neural_model.parameters()) + list(self.fusion_model.parameters()),
                            max_norm=1.0
                        )
                        self.scaler.step(optimizer)
                        self.scaler.update()
                        optimizer.zero_grad()
                else:
                    neural_pred = self.neural_model(batch_X)
                    fused_pred, confidence, fusion_weights = self.fusion_model(batch_X, neural_pred, symbolic_pred)

                    # Balanced loss function (0.5/0.5) with entropy regularization
                    neural_loss = criterion(neural_pred, batch_y)
                    fused_loss = criterion(fused_pred, batch_y)

                    # Add entropy regularization to prevent weight collapse
                    weight_entropy = -torch.mean(
                        fusion_weights[:, 0] * torch.log(fusion_weights[:, 0] + 1e-8) +
                        fusion_weights[:, 1] * torch.log(fusion_weights[:, 1] + 1e-8)
                    )
                    entropy_penalty = 0.1 * (0.693 - weight_entropy)  # 0.693 = max entropy for 2 weights

                    loss = (0.5 * neural_loss + 0.5 * fused_loss + entropy_penalty)
                    loss = loss / accumulation_steps

                    loss.backward()

                    if (batch_idx + 1) % accumulation_steps == 0:
                        torch.nn.utils.clip_grad_norm_(
                            list(self.neural_model.parameters()) + list(self.fusion_model.parameters()),
                            max_norm=1.0
                        )
                        optimizer.step()
                        optimizer.zero_grad()

                total_loss += loss.item() * accumulation_steps

                if show_progress:
                    batch_iterator.set_postfix({'loss': f'{loss.item() * accumulation_steps:.4f}'})

            scheduler.step()

            avg_loss = total_loss / num_batches
            if avg_loss < best_loss:
                best_loss = avg_loss
                improvement = "↓"
            else:
                improvement = "↑"

            if epoch % 10 == 0:
                current_lr = scheduler.get_last_lr()[0]
                print(f"Epoch {epoch:3d}/{epochs} | Loss: {avg_loss:.4f} {improvement} | LR: {current_lr:.6f}")

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        print(f"✓ Training complete! Best loss: {best_loss:.4f}")

        print("\n" + "─"*60)
        print("PHASE 4: Final Evaluation")
        print("─"*60)
        self._evaluate_final_model(X_tensor, y_tensor)
        self._print_validation_results()

    def _evaluate_final_model(self, X, y):
        """Evaluate final model performance."""
        self.neural_model.eval()
        self.fusion_model.eval()

        print("Evaluating model on full dataset...")
        with torch.no_grad():
            if self.use_amp:
                with torch.cuda.amp.autocast():
                    neural_pred = self.neural_model(X)
                    symbolic_pred = self.symbolic_predictions_cache
                    fused_pred, confidence, fusion_weights = self.fusion_model(X, neural_pred, symbolic_pred)
            else:
                neural_pred = self.neural_model(X)
                symbolic_pred = self.symbolic_predictions_cache
                fused_pred, confidence, fusion_weights = self.fusion_model(X, neural_pred, symbolic_pred)

            self.validation_metrics['final_fused'] = self._compute_metrics(
                y.cpu().numpy(),
                torch.argmax(fused_pred, dim=1).cpu().numpy(),
                fused_pred.cpu().numpy()
            )

            self.validation_metrics['final_neural'] = self._compute_metrics(
                y.cpu().numpy(),
                torch.argmax(neural_pred, dim=1).cpu().numpy(),
                neural_pred.cpu().numpy()
            )

            print(f"✓ Fusion weights - Neural: {torch.mean(fusion_weights[:, 0]).item():.3f}, "
                  f"Symbolic: {torch.mean(fusion_weights[:, 1]).item():.3f}")

    def _print_validation_results(self):
        """Print validation results."""
        print("\n" + "="*60)
        print("FINAL VALIDATION RESULTS")
        print("="*60)

        if 'final_fused' in self.validation_metrics:
            print("\nModel Comparison:")
            print(f"{'Metric':<15} {'Fused':<12} {'Neural':<12} {'Improvement':<12}")
            print("─"*60)

            for metric in ['accuracy', 'precision', 'recall', 'f1_score']:
                fused_val = getattr(self.validation_metrics['final_fused'], metric)
                neural_val = getattr(self.validation_metrics['final_neural'], metric)
                improvement = ((fused_val - neural_val) / neural_val * 100) if neural_val > 0 else 0
                symbol = "✓" if improvement > 0 else "✗"
                print(f"{metric.upper():<15} {fused_val:<12.4f} {neural_val:<12.4f} {improvement:>+11.2f}% {symbol}")

    def predict(self, symptoms: List[str], return_probabilities=False, return_explanation=False):
        """Make prediction for a given set of symptoms."""
        if self.neural_model is None or self.fusion_model is None:
            raise RuntimeError("Model not trained. Call train_model() first.")

        self.neural_model.eval()
        self.fusion_model.eval()

        X = self.symptom_scaler.transform(self.symptom_encoder.transform([symptoms]))
        X_tensor = torch.FloatTensor(X).to(self.device)

        with torch.no_grad():
            if self.use_amp:
                with torch.cuda.amp.autocast():
                    neural_pred = self.neural_model(X_tensor)
                    symbolic_pred = self._get_symbolic_predictions_batch(X_tensor)
                    fused_pred, confidence, fusion_weights = self.fusion_model(X_tensor, neural_pred, symbolic_pred)
            else:
                neural_pred = self.neural_model(X_tensor)
                symbolic_pred = self._get_symbolic_predictions_batch(X_tensor)
                fused_pred, confidence, fusion_weights = self.fusion_model(X_tensor, neural_pred, symbolic_pred)

        # Apply softmax to convert logits to probabilities
        fused_probs = torch.softmax(fused_pred, dim=1)

        predicted_idx = torch.argmax(fused_probs, dim=1).item()
        predicted_disease = self.disease_encoder.inverse_transform([predicted_idx])[0]

        result = {'disease': predicted_disease}

        if return_probabilities:
            probabilities = fused_probs[0].cpu().numpy()
            top_5_indices = np.argsort(probabilities)[-5:][::-1]
            result['top_5'] = [
                {
                    'disease': self.disease_encoder.inverse_transform([idx])[0],
                    'probability': float(probabilities[idx])
                }
                for idx in top_5_indices
            ]

        if return_explanation:
            result['confidence'] = float(confidence[0].item())
            result['neural_weight'] = float(fusion_weights[0, 0].item())
            result['symbolic_weight'] = float(fusion_weights[0, 1].item())

        return result

    def save_model(self, path: str):
        """Save trained model."""
        print(f"\nSaving model to {path}...")
        neo4j_config = None
        if hasattr(self.knowledge_graph, 'driver') and self.knowledge_graph.driver:
            neo4j_config = {
                'neo4j_uri': self.knowledge_graph.neo4j_uri,
                'neo4j_user': self.knowledge_graph.neo4j_user,
                'neo4j_password': self.knowledge_graph.neo4j_password
            }
            self.knowledge_graph.close()

        model_data = {
            'neural_model_state': self.neural_model.state_dict() if self.neural_model else None,
            'fusion_model_state': self.fusion_model.state_dict() if self.fusion_model else None,
            'symptom_encoder': self.symptom_encoder,
            'disease_encoder': self.disease_encoder,
            'symptom_scaler': self.symptom_scaler,
            'symptoms_vocab': self.symptoms_vocab,
            'diseases_vocab': self.diseases_vocab,
            'validation_metrics': self.validation_metrics,
            'knowledge_graph_rules': self.knowledge_graph.symptom_disease_rules,
            'neo4j_config': neo4j_config
        }

        with open(path, 'wb') as f:
            pickle.dump(model_data, f)

        file_size = Path(path).stat().st_size / 1e6
        print(f"✓ Model saved ({file_size:.2f} MB)")

        if neo4j_config:
            self.knowledge_graph.connect_to_neo4j()

    def load_model(self, path: str):
        """Load trained model from file."""
        with open(path, 'rb') as f:
            model_data = pickle.load(f)

        self.symptom_encoder = model_data['symptom_encoder']
        self.disease_encoder = model_data['disease_encoder']
        self.symptom_scaler = model_data['symptom_scaler']
        self.symptoms_vocab = model_data['symptoms_vocab']
        self.diseases_vocab = model_data['diseases_vocab']
        self.validation_metrics = model_data['validation_metrics']

        self.knowledge_graph = MedicalKnowledgeGraph()
        if 'knowledge_graph_rules' in model_data:
            self.knowledge_graph.symptom_disease_rules = model_data['knowledge_graph_rules']

        if model_data['neural_model_state']:
            self.neural_model = EnhancedNeuralPredictor(
                input_dim=len(self.symptoms_vocab),
                hidden_dims=[256, 192, 128],
                num_attention_heads=8,
                output_dim=len(self.diseases_vocab)
            ).to(self.device)
            self.neural_model.load_state_dict(model_data['neural_model_state'])

        if model_data['fusion_model_state']:
            self.fusion_model = CrossAttentionFusion(
                input_dim=len(self.symptoms_vocab),
                num_diseases=len(self.diseases_vocab)
            ).to(self.device)
            self.fusion_model.load_state_dict(model_data['fusion_model_state'])

        print(f"✓ Model loaded from {path}")

    def __del__(self):
        """Cleanup resources."""
        if hasattr(self, 'knowledge_graph') and self.knowledge_graph:
            self.knowledge_graph.close()

print("Disease predictor with mixed precision training loaded")

Disease predictor with mixed precision training loaded


---
## Section 6: Data Manager

Handles data extraction from Neo4j and training data generation.

In [10]:
class Neo4jDataManager:
    """Manages data extraction from Neo4j ontology for model training."""

    def __init__(self, neo4j_uri=NEO4J_URI,
                 neo4j_user=NEO4J_USER, neo4j_password=NEO4J_PASSWORD):
        self.neo4j_uri = neo4j_uri
        self.neo4j_user = neo4j_user
        self.neo4j_password = neo4j_password
        self.driver = None
        self.connect_to_neo4j()

    def connect_to_neo4j(self):
        """Establish connection to Neo4j database."""
        try:
            self.driver = GraphDatabase.driver(
                self.neo4j_uri,
                auth=(self.neo4j_user, self.neo4j_password)
            )
            with self.driver.session() as session:
                session.run("RETURN 1")
            logger.info("Connected to Neo4j")
            return True
        except Exception as e:
            logger.error(f"Failed to connect to Neo4j: {e}")
            return False

    def extract_training_data(self, min_weight=0.3, n_samples_per_disease=10):
        """Extract and generate training data from Neo4j ontology."""
        if not self.driver:
            return [], []

        try:
            with self.driver.session() as session:
                result = session.run("""
                    MATCH (d:Disease)-[r:HAS_SYMPTOM]->(s:Symptom)
                    WHERE r.weight >= $min_weight
                    RETURN d.name as disease, s.name as symptom, r.weight as weight
                    ORDER BY d.name, r.weight DESC
                """, min_weight=min_weight)

                disease_symptom_map = defaultdict(list)
                for record in result:
                    disease_symptom_map[record['disease']].append({
                        'symptom': record['symptom'],
                        'weight': record['weight']
                    })

                symptoms_data, diseases_data = [], []
                np.random.seed(42)

                for disease, symptom_weights in disease_symptom_map.items():
                    for _ in range(n_samples_per_disease):
                        case_symptoms = [
                            s['symptom'] for s in symptom_weights
                            if np.random.random() < min(0.9, s['weight'] * 1.5)
                        ]

                        if len(case_symptoms) < 2:
                            sorted_symptoms = sorted(symptom_weights,
                                                   key=lambda x: x['weight'],
                                                   reverse=True)
                            case_symptoms.extend([s['symptom'] for s in sorted_symptoms[:2]])

                        if case_symptoms:
                            symptoms_data.append(case_symptoms)
                            diseases_data.append(disease)

                logger.info(
                    f"Generated {len(symptoms_data)} samples from "
                    f"{len(set(diseases_data))} diseases"
                )
                return symptoms_data, diseases_data
        except Exception as e:
            logger.error(f"Failed to extract data: {e}")
            return [], []

    def close(self):
        """Close Neo4j connection."""
        if self.driver:
            self.driver.close()

print("Neo4jDataManager class loaded")

Neo4jDataManager class loaded


---
## Section 7: Training Pipeline

Define the training function that orchestrates the entire workflow.

In [11]:
def train_model(neo4j_uri=NEO4J_URI,
                min_weight=0.5,
                samples_per_disease=40,
                epochs=200,
                batch_size=64,
                accumulation_steps=4):
    """
    Training pipeline for neuro-symbolic disease prediction.

    Args:
        neo4j_uri: Neo4j connection URI
        min_weight: Minimum weight threshold for symptom relationships
        samples_per_disease: Number of training samples per disease
        epochs: Number of training epochs
        batch_size: Training batch size
        accumulation_steps: Gradient accumulation steps for effective larger batches

    Returns:
        Trained DiseasePredictor instance
    """
    print("Starting neuro-symbolic disease prediction model training")
    print("="*60)
    print(f"Training configuration:")
    print(f"  - Mixed precision training (FP16)")
    print(f"  - Gradient accumulation (steps={accumulation_steps})")
    print(f"  - Symbolic prediction caching")
    print(f"  - Model architecture [256, 192, 128] with attention")
    print(f"  - Batch size: {batch_size}")
    print("="*60)

    data_manager = Neo4jDataManager(neo4j_uri=neo4j_uri)
    predictor = DiseasePredictor()

    print("\nExtracting training data...")
    symptoms_data, diseases_data = data_manager.extract_training_data(
        min_weight=min_weight,
        n_samples_per_disease=samples_per_disease
    )

    if not symptoms_data:
        raise RuntimeError("No training data extracted")

    all_symptoms = set()
    for s in symptoms_data:
        all_symptoms.update(s)

    predictor.symptoms_vocab = sorted(list(all_symptoms))
    predictor.diseases_vocab = sorted(list(set(diseases_data)))

    disease_counts = Counter(diseases_data)
    filtered_data = [
        (s, d) for s, d in zip(symptoms_data, diseases_data)
        if disease_counts[d] >= 3
    ]
    symptoms_data, diseases_data = zip(*filtered_data)

    print(f"Dataset: {len(symptoms_data)} samples, "
          f"{len(predictor.symptoms_vocab)} symptoms, "
          f"{len(predictor.diseases_vocab)} diseases")

    print("\nLoading knowledge base...")
    predictor.load_knowledge_base_from_neo4j(min_weight=min_weight)

    print("\nTraining model...")
    predictor.train_model(
        list(symptoms_data),
        list(diseases_data),
        epochs=epochs,
        batch_size=batch_size,
        accumulation_steps=accumulation_steps
    )

    Path('models').mkdir(exist_ok=True)
    model_path = 'models/neo4j_disease_predictor.pkl'
    predictor.save_model(model_path)

    metadata = {
        'training_timestamp': datetime.datetime.now().isoformat(),
        'neo4j_uri': neo4j_uri,
        'min_weight': min_weight,
        'samples_per_disease': samples_per_disease,
        'epochs': epochs,
        'batch_size': batch_size,
        'accumulation_steps': accumulation_steps,
        'total_samples': len(symptoms_data),
        'pytorch_version': torch.__version__,
        'config': {
            'mixed_precision': True,
            'gradient_accumulation': True,
            'symbolic_caching': True,
            'architecture': '[256, 192, 128] with multi-head attention'
        }
    }

    with open(model_path.replace('.pkl', '_metadata.json'), 'w') as f:
        json.dump(metadata, f, indent=2)

    data_manager.close()
    print(f"\n{'='*60}")
    print(f"Training complete! Model saved to {model_path}")
    print(f"Training completed at: {datetime.datetime.now()}")
    print(f"{'='*60}")

    return predictor

print("Training function ready")

Training function ready


---
## Section 8: Execute Training

Run the training pipeline to train the neuro-symbolic disease predictor.

In [12]:
start_time = datetime.datetime.now()
print(f"Training started: {start_time}")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"GPU Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("\n" + "="*70)
print("OPTIMIZED TRAINING CONFIGURATION")
print("="*70)
print("Improvements:")
print("  • Samples per disease: 15 → 40 (167% more data)")
print("  • Training epochs: 100 → 200 (better convergence)")
print("  • Total samples: 27,270 → 72,720")
print("\nExpected Results:")
print("  • Accuracy: 23% → 50-65% (+127-178% improvement)")
print("  • Loss: 4.47 → 2.5-3.5 (-44-65% reduction)")
print("  • Medically relevant, diverse predictions")
print("  • Training time: ~25-30 minutes")
print("="*70 + "\n")

predictor = train_model(
    min_weight=0.5,
    samples_per_disease=40,  # ✅ Increased from 15 (167% more data)
    epochs=200,              # ✅ Increased from 100 (better convergence)
    batch_size=64,
    accumulation_steps=4
)

end_time = datetime.datetime.now()
duration = end_time - start_time
print(f"\nTraining completed: {end_time}")
print(f"Total duration: {duration}")

if torch.cuda.is_available():
    print(f"Peak GPU Memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

Training started: 2025-10-12 18:09:36.115793
Device: GPU
GPU Memory Available: 15.83 GB
Starting neuro-symbolic disease prediction model training
Training configuration:
  - Mixed precision training (FP16)
  - Gradient accumulation (steps=4)
  - Symbolic prediction caching
  - Model architecture [256, 192, 128] with attention
  - Batch size: 64
✓ Device: cuda, Mixed Precision: True

Extracting training data...
Dataset: 27270 samples, 2909 symptoms, 1818 diseases

Loading knowledge base...


Processing diseases:   0%|          | 0/1818 [00:00<?, ?it/s]

✓ Knowledge base loaded successfully

Training model...

────────────────────────────────────────────────────────────
PHASE 1: Data Encoding
────────────────────────────────────────────────────────────
✓ Encoded 27270 samples with 2909 features
✓ Transferred data to cuda

────────────────────────────────────────────────────────────
Pre-computing symbolic predictions...
────────────────────────────────────────────────────────────


Computing symbolic predictions:   0%|          | 0/27270 [00:00<?, ?it/s]

✓ Pre-computed 27270 symbolic predictions
✓ Cache size: 198.31 MB

────────────────────────────────────────────────────────────
PHASE 2: Model Initialization
────────────────────────────────────────────────────────────
✓ Neural model initialized: 1,318,362 parameters
✓ Fusion model initialized: 4,679,541 parameters
✓ Total parameters: 5,997,903
✓ AdamW optimizer configured (lr=0.001)
✓ Loss function and scheduler initialized

────────────────────────────────────────────────────────────
PHASE 3: Training
────────────────────────────────────────────────────────────
Configuration:
  • Epochs: 100
  • Batch size: 64
  • Accumulation steps: 4
  • Effective batch size: 256
  • Mixed precision: True
  • Symbolic predictions: cached
────────────────────────────────────────────────────────────


...

Epoch  70/100 | Loss: 4.6515 ↓ | LR: 0.000194


Epoch 81/100:   0%|          | 0/427 [00:00<?, ?it/s]

Epoch  80/100 | Loss: 4.5404 ↓ | LR: 0.000086


Epoch 91/100:   0%|          | 0/427 [00:00<?, ?it/s]

Epoch  90/100 | Loss: 4.4820 ↓ | LR: 0.000020
✓ Training complete! Best loss: 4.4674

────────────────────────────────────────────────────────────
PHASE 4: Final Evaluation
────────────────────────────────────────────────────────────
Evaluating model on full dataset...
✓ Fusion weights - Neural: 0.496, Symbolic: 0.504

FINAL VALIDATION RESULTS

Model Comparison:
Metric          Fused        Neural       Improvement 
────────────────────────────────────────────────────────────
ACCURACY        0.2339       0.4957            -52.81% ✗
PRECISION       0.0970       0.3400            -71.47% ✗
RECALL          0.2339       0.4957            -52.81% ✗
F1_SCORE        0.1224       0.3794            -67.74% ✗

Saving model to models/neo4j_disease_predictor.pkl...
✓ Model saved (25.37 MB)

Training complete! Model saved to models/neo4j_disease_predictor.pkl
Training completed at: 2025-10-12 18:19:11.222183

Training completed: 2025-10-12 18:19:11.225658
Total duration: 0:09:35.109865
Peak GPU Mem

---
## Section 9: Model Testing

Define testing functions with semantic symptom matching and execute comprehensive tests.

In [13]:
# Initialize semantic matcher for the trained model
print("Initializing semantic symptom matcher...")
semantic_matcher = SemanticSymptomMatcher(predictor.symptoms_vocab)

def predict_with_semantic_matching(predictor, user_symptoms, semantic_matcher,
                                   similarity_threshold=0.5, return_probabilities=True,
                                   return_explanation=True):
    """
    Enhanced prediction that handles colloquial symptom terms.

    Args:
        predictor: Trained DiseasePredictor instance
        user_symptoms: List of symptoms in any format (e.g., ["cough", "fever"])
        semantic_matcher: SemanticSymptomMatcher instance
        similarity_threshold: Minimum similarity score for matching (0-1)
        return_probabilities: Whether to return top-5 probabilities
        return_explanation: Whether to return fusion weights

    Returns:
        Dictionary with prediction results and matching information
    """
    print("\n" + "="*80)
    print("SEMANTIC SYMPTOM MATCHING")
    print("="*80)
    print(f"Input symptoms: {user_symptoms}\n")

    # Match symptoms using semantic similarity
    matches = semantic_matcher.match_symptoms_batch(
        user_symptoms,
        threshold=similarity_threshold,
        top_k_per_symptom=2
    )

    # Display matching results
    matched_symptoms = []
    print("Symptom Matching Results:")
    print(f"{'User Input':<30} {'Matched To':<30} {'Similarity':<12}")
    print("─"*80)

    for user_symptom, symptom_matches in matches.items():
        if symptom_matches:
            best_match, score = symptom_matches[0]
            matched_symptoms.append(best_match)
            status = "✓ Exact" if score == 1.0 else f"✓ {score:.2f}"
            print(f"{user_symptom:<30} {best_match:<30} {status:<12}")
        else:
            print(f"{user_symptom:<30} {'No match found':<30} {'✗':<12}")

    if not matched_symptoms:
        print("\nWARN: No symptoms could be matched!")
        print(f"Available symptoms (sample): {predictor.symptoms_vocab[:20]}")
        return None

    print(f"\nUsing {len(matched_symptoms)}/{len(user_symptoms)} matched symptoms for prediction\n")

    # Make prediction with matched symptoms
    if predictor.neural_model is None or predictor.fusion_model is None:
        raise RuntimeError("Model not trained")

    predictor.neural_model.eval()
    predictor.fusion_model.eval()

    X = predictor.symptom_scaler.transform(predictor.symptom_encoder.transform([matched_symptoms]))
    X_tensor = torch.FloatTensor(X).to(predictor.device)

    with torch.no_grad():
        if predictor.use_amp:
            with torch.cuda.amp.autocast():
                neural_logits = predictor.neural_model(X_tensor)
                symbolic_pred = predictor._get_symbolic_predictions_batch(X_tensor)
                fused_logits, confidence, fusion_weights = predictor.fusion_model(
                    X_tensor, neural_logits, symbolic_pred
                )
        else:
            neural_logits = predictor.neural_model(X_tensor)
            symbolic_pred = predictor._get_symbolic_predictions_batch(X_tensor)
            fused_logits, confidence, fusion_weights = predictor.fusion_model(
                X_tensor, neural_logits, symbolic_pred
            )

    # Convert logits to probabilities using softmax
    fused_probs = torch.softmax(fused_logits, dim=1)

    predicted_idx = torch.argmax(fused_probs, dim=1).item()
    predicted_disease = predictor.disease_encoder.inverse_transform([predicted_idx])[0]

    result = {
        'disease': predicted_disease,
        'matched_symptoms': matched_symptoms,
        'matching_info': matches
    }

    if return_probabilities:
        probabilities = fused_probs[0].cpu().numpy()
        top_5_indices = np.argsort(probabilities)[-5:][::-1]
        result['top_5'] = [
            {
                'disease': predictor.disease_encoder.inverse_transform([idx])[0],
                'probability': float(probabilities[idx])
            }
            for idx in top_5_indices
        ]

    if return_explanation:
        result['confidence'] = float(confidence[0].item())
        result['neural_weight'] = float(fusion_weights[0, 0].item())
        result['symbolic_weight'] = float(fusion_weights[0, 1].item())

    return result

print("✓ Semantic prediction function ready")

Initializing semantic symptom matcher...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/91 [00:00<?, ?it/s]

✓ Semantic prediction function ready


In [14]:
def run_model_tests(predictor):
    """Run comprehensive tests on the trained model."""

    print("\n" + "="*80)
    print("MODEL TESTING AND EVALUATION")
    print("="*80)

    # Test cases with various symptom combinations
    test_cases = [
        {
            'name': 'Respiratory Infection',
            'symptoms': ['cough', 'fever', 'fatigue', 'shortness of breath']
        },
        {
            'name': 'Cardiovascular Issue',
            'symptoms': ['chest pain', 'palpitations', 'dizziness', 'fatigue']
        },
        {
            'name': 'Gastrointestinal Problem',
            'symptoms': ['nausea', 'vomiting', 'abdominal pain', 'diarrhea']
        },
        {
            'name': 'Neurological Symptoms',
            'symptoms': ['headache', 'dizziness', 'confusion', 'weakness']
        },
        {
            'name': 'General Malaise',
            'symptoms': ['fatigue', 'weakness', 'loss of appetite']
        }
    ]

    results = []

    for i, test_case in enumerate(test_cases, 1):
        print(f"\n{'─'*80}")
        print(f"Test Case {i}: {test_case['name']}")
        print(f"{'─'*80}")
        print(f"Input Symptoms: {', '.join(test_case['symptoms'])}")
        print()

        try:
            # Make prediction with probabilities and explanation
            prediction = predictor.predict(
                test_case['symptoms'],
                return_probabilities=True,
                return_explanation=True
            )

            # Display results
            print(f"Predicted Disease: {prediction['disease']}")
            print(f"Model Confidence: {prediction['confidence']:.4f}")
            print(f"Neural Network Weight: {prediction['neural_weight']:.4f}")
            print(f"Symbolic Reasoning Weight: {prediction['symbolic_weight']:.4f}")

            print(f"\nTop 5 Predictions:")
            print(f"{'Rank':<6} {'Disease':<40} {'Probability':<12}")
            print("─"*60)
            for rank, pred in enumerate(prediction['top_5'], 1):
                print(f"{rank:<6} {pred['disease']:<40} {pred['probability']:<12.6f}")

            results.append({
                'test_case': test_case['name'],
                'symptoms': test_case['symptoms'],
                'prediction': prediction
            })

        except Exception as e:
            print(f"Error during prediction: {e}")
            results.append({
                'test_case': test_case['name'],
                'symptoms': test_case['symptoms'],
                'error': str(e)
            })

    # Summary statistics
    print(f"\n{'='*80}")
    print("TEST SUMMARY")
    print(f"{'='*80}")

    successful_tests = [r for r in results if 'error' not in r]
    failed_tests = [r for r in results if 'error' in r]

    print(f"Total Test Cases: {len(test_cases)}")
    print(f"Successful Predictions: {len(successful_tests)}")
    print(f"Failed Predictions: {len(failed_tests)}")

    if successful_tests:
        avg_confidence = np.mean([r['prediction']['confidence'] for r in successful_tests])
        avg_neural_weight = np.mean([r['prediction']['neural_weight'] for r in successful_tests])
        avg_symbolic_weight = np.mean([r['prediction']['symbolic_weight'] for r in successful_tests])

        print(f"\nAverage Model Confidence: {avg_confidence:.4f}")
        print(f"Average Neural Weight: {avg_neural_weight:.4f}")
        print(f"Average Symbolic Weight: {avg_symbolic_weight:.4f}")

        fusion_balance = "Neural-dominant" if avg_neural_weight > avg_symbolic_weight else "Symbolic-dominant"
        print(f"Fusion Strategy: {fusion_balance}")

    print(f"{'='*80}\n")

    return results

print("Model testing function ready")

Model testing function ready


In [15]:
# Run tests on the trained model
print("\nRunning model tests...")
test_results = run_model_tests(predictor)

# Save test results
test_results_path = 'models/test_results.json'
with open(test_results_path, 'w') as f:
    json.dump(test_results, f, indent=2, default=str)
print(f"\nTest results saved to {test_results_path}")


Running model tests...

MODEL TESTING AND EVALUATION

────────────────────────────────────────────────────────────────────────────────
Test Case 1: Respiratory Infection
────────────────────────────────────────────────────────────────────────────────
Input Symptoms: cough, fever, fatigue, shortness of breath

Predicted Disease: palatal paralysis
Model Confidence: 0.5142
Neural Network Weight: 0.4994
Symbolic Reasoning Weight: 0.5006

Top 5 Predictions:
Rank   Disease                                  Probability 
────────────────────────────────────────────────────────────
1      palatal paralysis                        0.009912    
2      mercury and its compounds induced dermatopathy 0.009593    
3      pediatric obsessive-compulsive disorder (ocd) 0.009537    
4      persistent junctional reciprocating tachycardia (pjrt) 0.009124    
5      osteoporosis                             0.009072    

────────────────────────────────────────────────────────────────────────────────
Test Cas

---
## Section 10: Save Results

Save trained model and test results to Google Drive.

In [16]:
!cp -r /content/models /content/drive/MyDrive/MSc/NeuroSymbolicPredictor